# Migracao Oracle → ClickHouse

**IMPORTANTE**: Feche o Jupyter completamente e execute no terminal:
```bash
pip install "numpy<2" pandas --force-reinstall
```
Depois abra o Jupyter novamente.

## 1. Verificar NumPy

In [1]:
import numpy as np
print(f"NumPy: {np.__version__}")

NumPy: 1.26.4


## 2. Criar Spark Session (COM Arrow desabilitado)

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("oracle-clickhouse-migration")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark Session criada (Arrow desabilitado)")

Spark Session criada (Arrow desabilitado)


## 3. Configurar ClickHouse JDBC

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("oracle-clickhouse-migration")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark Session criada (Arrow desabilitado)")

Spark Session criada (Arrow desabilitado)


In [4]:
clickhouse_host = "e1a1lieug8.us-central1.gcp.clickhouse.cloud"
clickhouse_port = 8443
clickhouse_user = "default"
clickhouse_password = "_uv765EvWphL_"
clickhouse_database = "raw"

clickhouse_jdbc_url = f"jdbc:clickhouse://{clickhouse_host}:{clickhouse_port}/{clickhouse_database}?ssl=true"

print(f"ClickHouse JDBC URL: {clickhouse_jdbc_url}")

clickhouse_jdbc_props = {
    "user": clickhouse_user,
    "password": clickhouse_password,
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "ssl": "true",
    "sslmode": "strict"
}

print("ClickHouse configurado para uso com Spark JDBC")

ClickHouse JDBC URL: jdbc:clickhouse://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/raw?ssl=true
ClickHouse configurado para uso com Spark JDBC


## 4. Configurar Oracle


In [5]:
oracle_host = "10.255.150.11"
oracle_port = 1521
oracle_service = "bi.grupotracker.com.br"
oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
jdbc_opts = {
    "url": jdbc_url,
    "user": oracle_user,
    "password": oracle_password,
    "driver": "oracle.jdbc.OracleDriver"
}
print(f"JDBC URL: {jdbc_url}")

JDBC URL: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br


## 5. Teste de Conexão Oracle


In [6]:
df_test = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", oracle_user)
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("query", "SELECT 1 AS ok FROM dual")
    .load()
)

df_test.show()
print("[OK] Oracle funcionando")

+------------+
|          OK|
+------------+
|1.0000000000|
+------------+

[OK] Oracle funcionando


## 6. Definir Tabelas para Migração


In [11]:
tables_to_extract = [
    "ginf.depara_cliente",
    "ginf.BASE_CEP_COMPLETA",
    "ginf.TST_CONTRATOS_BI",
    "ginf.BASE_REGIONAL",
    "ginf.TAB_CIDADE_DELITO_SP_CAP",
    "siga.SC5030",
    "siga.SC6030",
    "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS",
    "siga.SD2030",
    "siga.CN9030",
    "siga.SA1030",
    "siga.SA3030",
    "siga.SB1030",
    "siga.SZH030",
    "siga.SZJ030",
    "siga.SZU030",
    "siga.SZV030",
    "siga.SZW030",
    "siga.ZAA030",
    "siga.ZA1030",
    "siga.ZA3030",
    "siga.ZB3030",
    "siga.ZE8030",
    "siga.ZTX030",
    "siga.ZT1030",
    "siga.CN1030",
    "siga.CNB030",
    "siga.SE4030",
    "siga.SF2030",
    "ginf.TST_CONTRATOS",
    "scot.ERP_PRODUCT",
    "scot.ERP_PRODUCT_ITEM",
    "scot.ERP_VEHICLE",
    "scot.ERP_AGREEMENT",
    "scot.SC_CITY",
    "scot.SC_GROUP",
    "scot.SC_LOCATION",
    "scot.SC_REQ_FILE",
    "scot.SC_REQUISITION",
    "scot.SC_REQUISITION_HISTORY",
    "scot.SC_REQUISITION_QUEUE",
    "scot.SC_REQUISITION_STATUS",
    "scot.SC_RESERVE",
    "scot.SC_RESERVE_LOCATION",
    "scot.SC_RESULT_CODE",
    "scot.SC_ROLE",
    "scot.SC_STATE",
    "scot.CEPREG",
    "scot.SC_TASK",
    "scot.SC_WAREHOUSE",
    "scot.SC_TECHNICAL_REGISTER",
    "scot.SC_WEBSERVICE_REQUISITION",
    "scot.SC_WEBSERVICE_REQUISITION_HISTORY"
]

# Gerar dicionário ch_tables automaticamente (remove duplicatas)
ch_tables = {}
seen = set()

for table in tables_to_extract:
    schema, name = table.split(".", 1)
    ch_name = name.lower()

    # Evitar duplicatas
    if ch_name not in seen:
        ch_tables[ch_name] = table
        seen.add(ch_name)

print(f"Total de tabelas únicas: {len(ch_tables)}")
print(f"\nPrimeiras 5 tabelas:")
for i, (key, value) in enumerate(list(ch_tables.items())[:5]):
    print(f"  {key} -> {value}")

Total de tabelas únicas: 54

Primeiras 5 tabelas:
  depara_cliente -> ginf.depara_cliente
  base_cep_completa -> ginf.BASE_CEP_COMPLETA
  tst_contratos_bi -> ginf.TST_CONTRATOS_BI
  base_regional -> ginf.BASE_REGIONAL
  tab_cidade_delito_sp_cap -> ginf.TAB_CIDADE_DELITO_SP_CAP


In [15]:
# Verificar quais tabelas já têm dados no ClickHouse (banco raw) e filtrar apenas as vazias

print("=" * 80)
print("VERIFICANDO TABELAS COM E SEM DADOS NO CLICKHOUSE (BANCO RAW)")
print("=" * 80)

tables_with_data = []
tables_without_data = []
tables_not_exist = []

for ch_tbl, oracle_tbl in ch_tables.items():
    try:
        # Tentar contar linhas na tabela ClickHouse no banco RAW
        count_query = f"SELECT COUNT(*) as cnt FROM raw.{ch_tbl}"
        result = client.query(count_query).result_rows

        if result:
            row_count = result[0][0]

            if row_count > 0:
                tables_with_data.append((ch_tbl, oracle_tbl, row_count))
                print(f"✅ raw.{ch_tbl:40s} -> {row_count:>15,} linhas")
            else:
                tables_without_data.append((ch_tbl, oracle_tbl))
                print(f"⚪ raw.{ch_tbl:40s} -> VAZIA (0 linhas)")
    except Exception as e:
        # Tabela não existe
        tables_not_exist.append((ch_tbl, oracle_tbl))
        print(f"❌ raw.{ch_tbl:40s} -> NÃO EXISTE")

print("\n" + "=" * 80)
print("RESUMO")
print("=" * 80)
print(f"Tabelas COM dados:     {len(tables_with_data)}")
print(f"Tabelas VAZIAS:        {len(tables_without_data)}")
print(f"Tabelas NÃO EXISTEM:   {len(tables_not_exist)}")

print("\n" + "=" * 80)
print("TABELAS SEM DADOS (VAZIAS + NÃO EXISTEM)")
print("=" * 80)

# Combinar tabelas vazias e que não existem
tables_to_migrate = tables_without_data + tables_not_exist

if tables_to_migrate:
    for ch_tbl, oracle_tbl in tables_to_migrate:
        print(f"  • {oracle_tbl} -> raw.{ch_tbl}")

    # Criar dicionário apenas com tabelas sem dados
    ch_tables_empty = {ch_tbl: oracle_tbl for ch_tbl, oracle_tbl in tables_to_migrate}

    print(f"\n Total de tabelas para migrar: {len(ch_tables_empty)}")
    print("\nPara migrar apenas estas tabelas, use:")
    print("  ch_tables = ch_tables_empty")
else:
    print("✅ Todas as tabelas já têm dados no ClickHouse (raw)!")

print("=" * 80)


VERIFICANDO TABELAS COM E SEM DADOS NO CLICKHOUSE (BANCO RAW)
✅ raw.depara_cliente                           ->              47 linhas
✅ raw.base_cep_completa                        ->          96,778 linhas
✅ raw.tst_contratos_bi                         ->         100,000 linhas
✅ raw.base_regional                            ->              27 linhas
✅ raw.tab_cidade_delito_sp_cap                 ->              45 linhas
✅ raw.sc5030                                   ->          50,000 linhas
✅ raw.sc6030                                   ->          50,000 linhas
✅ raw.tst_historico_solicitacoes               ->         100,000 linhas
✅ raw.tst_solicit_cadastradas                  ->          50,130 linhas
✅ raw.sd2030                                   ->          50,000 linhas
✅ raw.cn9030                                   ->         100,000 linhas
✅ raw.sa1030                                   ->         100,000 linhas
✅ raw.sa3030                                   ->           3,

## 7. Migração Inicial - RAW/Bronze Layer

In [14]:
import clickhouse_connect
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Manter cliente ClickHouse para gerenciar progresso
client = clickhouse_connect.get_client(
    host=clickhouse_host,
    port=8443,
    username=clickhouse_user,
    password=clickhouse_password,
    database=clickhouse_database,
    secure=True,
    connect_timeout=60,
    send_receive_timeout=300
)
print("ClickHouse client OK (para gerenciar progresso)\n")

# Limite de linhas por batch
MAX_ROWS = 50000

# Criar tabela de progresso no ClickHouse
try:
    client.command("""
        CREATE TABLE IF NOT EXISTS migration_progress
        (
            oracle_table String,
            ch_table String,
            rows_collected Int64,
            total_rows Float64,
            rows_remaining Float64,
            last_id String,
            id_column String,
            status String,
            error String,
            updated_at DateTime DEFAULT now()
        ) ENGINE = ReplacingMergeTree(updated_at)
        ORDER BY oracle_table
    """)
    print("[INFO] Tabela de progresso criada/verificada\n")
except Exception as e:
    print(f"[AVISO] Erro ao criar tabela de progresso: {str(e)[:100]}\n")


def get_oracle_column_metadata(oracle_tbl):
    """Busca metadados da tabela Oracle para orientar casts e tipagem no ClickHouse."""
    try:
        owner, table_name = oracle_tbl.split('.', 1)
        metadata_query = f"""
            SELECT
                COLUMN_NAME,
                DATA_TYPE,
                DATA_PRECISION,
                DATA_SCALE,
                DATA_LENGTH
            FROM ALL_TAB_COLUMNS
            WHERE OWNER = '{owner.upper()}'
              AND TABLE_NAME = '{table_name.upper()}'
            ORDER BY COLUMN_ID
        """
        meta_df = (
            spark.read.format("jdbc")
            .options(**jdbc_opts)
            .option("dbtable", f"({metadata_query}) tmp")
            .load()
        )

        metadata = {}
        for row in meta_df.collect():
            metadata[row['COLUMN_NAME']] = {
                'data_type': row['DATA_TYPE'],
                'data_precision': row['DATA_PRECISION'],
                'data_scale': row['DATA_SCALE'],
                'data_length': row['DATA_LENGTH']
            }
        return metadata
    except Exception as e:
        print(f"  [AVISO] Nao foi possivel ler metadados Oracle de {oracle_tbl}: {str(e)[:100]}")
        return {}


def oracle_type_to_spark_cast(meta):
    """Mapeia tipo Oracle para cast Spark equivalente."""
    if not meta:
        return None

    data_type = str(meta.get('data_type') or '').upper()
    precision = meta.get('data_precision')
    scale = meta.get('data_scale')

    if data_type in {'NUMBER', 'DECIMAL', 'NUMERIC'}:
        p = int(precision) if precision is not None else 38
        s = int(scale) if scale is not None else 0
        p = max(1, min(p, 38))
        s = max(0, min(s, p))

        if s == 0:
            if p <= 9:
                return 'int'
            if p <= 18:
                return 'bigint'
        return f'decimal({p},{s})'

    if data_type in {'FLOAT', 'BINARY_FLOAT', 'BINARY_DOUBLE'}:
        return 'double'

    if data_type in {'INTEGER', 'SMALLINT'}:
        return 'int'

    if data_type == 'DATE' or data_type.startswith('TIMESTAMP'):
        return 'timestamp'

    if data_type in {'CHAR', 'NCHAR', 'VARCHAR2', 'NVARCHAR2', 'CLOB', 'NCLOB', 'LONG', 'XMLTYPE', 'RAW', 'LONG RAW', 'BLOB'}:
        return 'string'

    return None


def apply_oracle_casts(df, oracle_metadata):
    """Aplica CAST com base no tipo de origem Oracle apenas quando houver mapeamento."""
    if not oracle_metadata:
        return df

    cast_exprs = []
    casted = []

    for col_name in df.columns:
        meta = oracle_metadata.get(col_name) or oracle_metadata.get(col_name.upper())
        target_cast = oracle_type_to_spark_cast(meta)

        if target_cast:
            cast_exprs.append(F.col(f"`{col_name}`").cast(target_cast).alias(col_name))
            casted.append((col_name, str(meta.get('data_type')), target_cast))
        else:
            cast_exprs.append(F.col(f"`{col_name}`"))

    if casted:
        preview = ', '.join([f"{c}({src}->{dst})" for c, src, dst in casted[:8]])
        print(f"  [CAST] {len(casted)} coluna(s) com cast por tipo Oracle. Ex.: {preview}")

    return df.select(*cast_exprs)


def spark_to_ch_type(spark_type):
    """Converte tipo Spark para tipo ClickHouse preservando numericos e datas."""
    tipo = str(spark_type).lower()

    if tipo.startswith('decimal(') and tipo.endswith(')'):
        inner = tipo[len('decimal('):-1]
        parts = [p.strip() for p in inner.split(',')]
        if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
            return f"Nullable(Decimal({parts[0]},{parts[1]}))"
        return 'Nullable(Decimal(38,10))'

    if 'bigint' in tipo or 'long' in tipo:
        return 'Nullable(Int64)'

    if tipo in {'int', 'integer'} or 'int' in tipo:
        return 'Nullable(Int32)'

    if 'double' in tipo:
        return 'Nullable(Float64)'

    if 'float' in tipo:
        return 'Nullable(Float32)'

    if 'boolean' in tipo:
        return 'Nullable(UInt8)'

    if tipo == 'date':
        return 'Nullable(Date)'

    if 'timestamp' in tipo:
        return 'Nullable(DateTime64(3))'

    if 'binary' in tipo:
        return 'Nullable(String)'

    return 'Nullable(String)'


def create_clickhouse_table_from_spark(df, ch_tbl):
    """Cria tabela no ClickHouse baseada no schema do DataFrame Spark"""
    ch_cols = []
    for field in df.schema.fields:
        ch_type = spark_to_ch_type(field.dataType)
        ch_cols.append(f"`{field.name}` {ch_type}")

    create_sql = f"""
        CREATE TABLE IF NOT EXISTS {ch_tbl} (
            {', '.join(ch_cols)}
        ) ENGINE = MergeTree()
        ORDER BY tuple()
    """

    client.command(create_sql)
    return True


def remove_duplicates_spark(df, id_col):
    """
    [RAW/BRONZE LAYER] Mantém dados como estão - SEM REMOÇÃO DE DUPLICATAS

    Na arquitetura Medallion:
    - BRONZE/RAW: Dados brutos do Oracle, exatamente como estão (incluindo duplicatas)
    - SILVER: Dados limpos e transformados (onde a remoção de duplicatas deve ocorrer)
    - GOLD: Dados agregados e prontos para análise

    Esta função retorna o DataFrame original sem modificações.
    """
    print("  [RAW] Dados mantidos como estão (incluindo duplicatas)")
    return df

    # ---- CÓDIGO COMENTADO PARA USO NA SILVER LAYER ----
    # Para remover duplicatas na camada SILVER, descomente o código abaixo:
    #
    if not id_col or id_col not in df.columns:
        # Se não houver coluna ID, usar distinct() em todas as colunas
        original_count = df.count()
        df_clean = df.distinct()
        clean_count = df_clean.count()
        if original_count > clean_count:
            print(f"  [SILVER] {original_count - clean_count} duplicata(s) exata(s) removida(s)")
        return df_clean

    # Usar window function para manter apenas a primeira ocorrência de cada ID
    window_spec = Window.partitionBy(id_col).orderBy(F.monotonically_increasing_id())
    df_dedup = df.withColumn("row_num", F.row_number().over(window_spec)) \
                 .filter(F.col("row_num") == 1) \
                 .drop("row_num")

    original_count = df.count()
    dedup_count = df_dedup.count()

    if original_count > dedup_count:
        print(f"  [SILVER] {original_count - dedup_count} duplicata(s) removida(s)")

    return df_dedup


def save_progress_to_ch(oracle_tbl, ch_tbl, stats):
    """Salva progresso no ClickHouse usando pandas"""
    try:
        progress_data = pd.DataFrame([{
            'oracle_table': oracle_tbl,
            'ch_table': ch_tbl,
            'rows_collected': int(stats.get('rows_collected', 0)),
            'total_rows': float(stats.get('total_rows', 0)),
            'rows_remaining': float(stats.get('rows_remaining', 0)),
            'last_id': str(stats.get('last_id', '')),
            'id_column': str(stats.get('id_column', '')),
            'status': stats.get('status', 'unknown'),
            'error': str(stats.get('error', ''))[:500]
        }])
        client.insert_df('migration_progress', progress_data)
    except Exception as e:
        print(f"  [AVISO] Erro ao salvar progresso: {str(e)[:80]}")


def get_rows_in_ch(ch_tbl):
    """Retorna quantidade de linhas no ClickHouse ou 0 se tabela nao existe"""
    try:
        r = client.query(f"SELECT count() as cnt FROM {ch_tbl}").result_rows
        return int(r[0][0]) if r else 0
    except Exception:
        return 0


def get_last_progress(oracle_tbl):
    """Retorna ultimo progresso da tabela (last_id, id_column)"""
    try:
        r = client.query(
            f"SELECT last_id, id_column FROM migration_progress "
            f"WHERE oracle_table = '{oracle_tbl}' ORDER BY updated_at DESC LIMIT 1"
        ).result_rows
        return (r[0][0], r[0][1]) if r else (None, None)
    except Exception:
        return (None, None)


sep = "=" * 60
print(sep)
print(f"MIGRAÇÃO ORACLE → CLICKHOUSE VIA SPARK")
print(f"Batch size: {MAX_ROWS:,} linhas")
print(sep)

start = datetime.now()
ok = 0
fail = 0
total_rows = 0
failed = []
migration_stats = {}



# Mapear tabelas (lista completa para migração inicial RAW/Bronze)
tables_to_extract = [
    "ginf.depara_cliente",
    "ginf.BASE_CEP_COMPLETA",
    "bistage.TST_CONTRATOS_BI",
    "ginf.BASE_REGIONAL",
    "ginf.TAB_CIDADE_DELITO_SP_CAP",
    "siga.SC5030",
    "siga.SC6030",
    "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS",
    "siga.SD2030",
    "siga.CN9030",
    "siga.SA1030",
    "siga.SA3030",
    "siga.SB1030",
    "siga.SZH030",
    "siga.SZJ030",
    "siga.SZU030",
    "siga.SZV030",
    "siga.SZW030",
    "siga.ZAA030",
    "siga.ZA1030",
    "siga.ZA3030",
    "siga.ZB3030",
    "siga.ZE8030",
    "siga.ZTX030",
    "siga.ZT1030",
    "siga.CN1030",
    "siga.CNB030",
    "siga.SE4030",
    "siga.SF2030",
    "ginf.TST_CONTRATOS",
    "scot.ERP_PRODUCT",
    "scot.ERP_PRODUCT_ITEM",
    "scot.ERP_VEHICLE",
    "scot.ERP_AGREEMENT",
    "scot.SC_CITY",
    "scot.SC_GROUP",
    "scot.SC_LOCATION",
    "scot.SC_REQ_FILE",
    "scot.SC_REQUISITION",
    "scot.SC_REQUISITION_HISTORY",
    "scot.SC_REQUISITION_QUEUE",
    "scot.SC_REQUISITION_STATUS",
    "scot.SC_RESERVE",
    "scot.SC_RESERVE_LOCATION",
    "scot.SC_RESULT_CODE",
    "scot.SC_ROLE",
    "scot.SC_STATE",
    "scot.CEPREG",
    "scot.SC_TASK",
    "scot.SC_WAREHOUSE",
    "scot.SC_TECHNICAL_REGISTER",
    "scot.SC_WEBSERVICE_REQUISITION",
    "scot.SC_WEBSERVICE_REQUISITION_HISTORY"
]
ch_tables = {}
seen = set()
for table in tables_to_extract:
    schema, name = table.split(".", 1)
    ch_name = name.lower()
    if ch_name not in seen:
        ch_tables[ch_name] = table
        seen.add(ch_name)

for i, (ch_tbl, oracle_tbl) in enumerate(ch_tables.items(), 1):
    pct = (i / len(ch_tables)) * 100
    print(f"\n[{i}/{len(ch_tables)}] ({pct:.1f}%) {oracle_tbl} -> {ch_tbl}")

    try:
        t0 = datetime.now()

        # 1. CONTAR linhas no Oracle e no ClickHouse
        try:
            df_count = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {oracle_tbl}) tmp") \
                .load()
            total_oracle_rows = float(df_count.collect()[0]['CNT'])
            print(f"  Total no Oracle: {total_oracle_rows:,.0f} linhas")
        except Exception as count_err:
            print(f"  [ERRO] Falha ao contar: {str(count_err)[:100]}")
            stats = {
                "status": "failed",
                "error": f"Count failed: {str(count_err)[:100]}",
                "rows_collected": 0,
                "total_rows": 0,
                "rows_remaining": 0,
                "last_id": "",
                "id_column": ""
            }
            migration_stats[oracle_tbl] = stats
            save_progress_to_ch(oracle_tbl, ch_tbl, stats)
            fail += 1
            failed.append(oracle_tbl)
            continue

        rows_in_ch = get_rows_in_ch(ch_tbl)
        rows_remaining = max(0, total_oracle_rows - rows_in_ch)
        print(f"  No ClickHouse: {rows_in_ch:,.0f} linhas | Faltam: {rows_remaining:,.0f}")

        if rows_remaining <= 0:
            print(f"  [SKIP] Tabela completa")
            ok += 1
            continue

        if rows_in_ch >= MAX_ROWS:
            print(f"  [SKIP] ClickHouse ja tem >= {MAX_ROWS:,} linhas, nao traz mais")
            ok += 1
            continue

        do_overwrite = rows_remaining < MAX_ROWS
        if do_overwrite:
            print(f"  [MODO] Overwrite (faltam {rows_remaining:,.0f} < {MAX_ROWS:,})")
        else:
            print(f"  [MODO] Append (faltam {rows_remaining:,.0f} >= {MAX_ROWS:,})")

        # 2. LER dados do Oracle
        df = None
        id_col = None
        last_id_progress, id_col_progress = get_last_progress(oracle_tbl) if not do_overwrite else (None, None)

        if do_overwrite:
            try:
                client.command(f"TRUNCATE TABLE IF EXISTS {ch_tbl}")
                print(f"  [OK] Tabela truncada")
            except Exception as trunc_err:
                print(f"  [AVISO] Truncate falhou (tabela pode nao existir): {str(trunc_err)[:80]}")

        df_sample = spark.read.format("jdbc").options(**jdbc_opts).option("dbtable", f"(SELECT * FROM {oracle_tbl} WHERE ROWNUM <= 1) tmp").load()
        cols = df_sample.columns
        id_col = None
        for c in cols:
            if 'ID' in c.upper() or c == cols[0]:
                id_col = c
                break
        id_col = id_col or cols[0]

        oracle_metadata = get_oracle_column_metadata(oracle_tbl)
        if oracle_metadata:
            print(f"  [INFO] Metadados Oracle carregados: {len(oracle_metadata)} coluna(s)")

        def _build_query(last_id_val):
            if last_id_val is None or (isinstance(last_id_val, str) and not str(last_id_val).strip()):
                return f"(SELECT * FROM {oracle_tbl} ORDER BY {id_col} FETCH FIRST {MAX_ROWS} ROWS ONLY) tmp"
            quote = "'" if isinstance(last_id_val, str) else ""
            return f"(SELECT * FROM {oracle_tbl} WHERE {id_col} > {quote}{last_id_val}{quote} ORDER BY {id_col} FETCH FIRST {MAX_ROWS} ROWS ONLY) tmp"

        nrows_total = 0
        last_id = last_id_progress if not do_overwrite else None
        batch_num = 0

        while True:
            batch_num += 1
            dbtable = _build_query(last_id)
            print(f"  Batch {batch_num}: lendo ate {MAX_ROWS:,} linhas...")
            df = spark.read.format("jdbc").options(**jdbc_opts).option("dbtable", dbtable).load()
            df = apply_oracle_casts(df, oracle_metadata)
            nrows = df.count()

            if nrows == 0:
                if batch_num == 1:
                    print(f"  [AVISO] Tabela vazia")
                    stats = {"rows_collected": 0, "total_rows": total_oracle_rows, "rows_remaining": total_oracle_rows, "last_id": "", "id_column": "", "status": "empty", "error": ""}
                    migration_stats[oracle_tbl] = stats
                    save_progress_to_ch(oracle_tbl, ch_tbl, stats)
                    fail += 1
                    failed.append(oracle_tbl)
                break

            nrows_total += nrows
            try:
                client.command(f"SELECT 1 FROM {ch_tbl} LIMIT 1")
            except Exception:
                create_clickhouse_table_from_spark(df, ch_tbl)
                print(f"  Tabela '{ch_tbl}' criada")

            df.write.format("jdbc").option("url", clickhouse_jdbc_url).option("dbtable", ch_tbl).option("user", clickhouse_user).option("password", clickhouse_password).option("driver", "com.clickhouse.jdbc.ClickHouseDriver").option("batchsize", 10000).option("isolationLevel", "NONE").mode("append").save()

            last_row = df.orderBy(F.col(id_col).desc()).first()
            last_id = last_row[id_col] if last_row else None

            if not do_overwrite:
                break
            if nrows < MAX_ROWS:
                break

        if nrows_total == 0 and batch_num == 1:
            continue

        dur = (datetime.now() - t0).total_seconds()
        rows_remaining_final = max(0, total_oracle_rows - get_rows_in_ch(ch_tbl))

        print(f"  [OK] {nrows_total:,} linhas inseridas em {dur:.2f}s")
        print(f"  Restante: {rows_remaining_final:,.0f} linhas")
        if last_id:
            print(f"  Ultimo ID: {last_id}")

        stats = {
            "rows_collected": nrows_total,
            "total_rows": total_oracle_rows,
            "rows_remaining": rows_remaining_final,
            "last_id": str(last_id) if last_id else "",
            "id_column": id_col or "",
            "status": "complete" if rows_remaining_final == 0 else "partial",
            "error": ""
        }
        migration_stats[oracle_tbl] = stats
        save_progress_to_ch(oracle_tbl, ch_tbl, stats)

        ok += 1
        total_rows += nrows_total

    except Exception as e:
        msg = str(e)[:200]
        print(f"  [ERRO] {msg}")
        fail += 1
        failed.append(oracle_tbl)
        stats = {
            "status": "failed",
            "error": msg,
            "rows_collected": 0,
            "total_rows": 0,
            "rows_remaining": 0,
            "last_id": "",
            "id_column": ""
        }
        migration_stats[oracle_tbl] = stats
        save_progress_to_ch(oracle_tbl, ch_tbl, stats)

dur = (datetime.now() - start).total_seconds()

print(f"\n{sep}")
print("RESUMO")
print(sep)
print(f"Sucesso: {ok}/{len(ch_tables)}")
print(f"Falhas: {fail}")
print(f"Total linhas coletadas: {total_rows:,}")
print(f"Tempo: {dur:.2f}s")

if failed:
    print(f"\nTabelas com erro:")
    for t in failed:
        print(f"  - {t}")

print(f"\n{sep}")
print("[INFO] Progresso salvo no migration_progress")
print(sep)

ClickHouse client OK (para gerenciar progresso)

[INFO] Tabela de progresso criada/verificada

MIGRAÇÃO ORACLE → CLICKHOUSE VIA SPARK
Batch size: 50,000 linhas

[1/54] (1.9%) ginf.depara_cliente -> depara_cliente
  Total no Oracle: 47 linhas
  No ClickHouse: 47 linhas | Faltam: 0
  [SKIP] Tabela completa

[2/54] (3.7%) ginf.BASE_CEP_COMPLETA -> base_cep_completa
  Total no Oracle: 96,778 linhas
  No ClickHouse: 96,778 linhas | Faltam: 0
  [SKIP] Tabela completa

[3/54] (5.6%) bistage.TST_CONTRATOS_BI -> tst_contratos_bi
  Total no Oracle: 2,926,606 linhas
  No ClickHouse: 100,000 linhas | Faltam: 2,826,606
  [SKIP] ClickHouse ja tem >= 50,000 linhas, nao traz mais

[4/54] (7.4%) ginf.BASE_REGIONAL -> base_regional
  Total no Oracle: 27 linhas
  No ClickHouse: 27 linhas | Faltam: 0
  [SKIP] Tabela completa

[5/54] (9.3%) ginf.TAB_CIDADE_DELITO_SP_CAP -> tab_cidade_delito_sp_cap
  Total no Oracle: 45 linhas
  No ClickHouse: 45 linhas | Faltam: 0
  [SKIP] Tabela completa

[6/54] (11.1%) s

## 8. Migração Incremental - Continuar de onde parou

In [ ]:
## 8. Migração Incremental - Continuar de onde parou

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime
import time

print("=" * 60)
print("MIGRAÇÃO INCREMENTAL COMPLETA - ORACLE → CLICKHOUSE")
print("=" * 60)
MAX_ROWS = 1000000
# Configurações de performance
MAX_ITERATIONS = 1000  # Máximo de iterações para evitar loop infinito
SLEEP_BETWEEN_BATCHES = 2  # Segundos entre batches (evitar sobrecarga)

iteration = 0
total_migrated = 0

while iteration < MAX_ITERATIONS:
    iteration += 1
    
    print(f"\n{'='*60}")
    print(f"ITERAÇÃO {iteration}/{MAX_ITERATIONS}")
    print(f"{'='*60}")
    
    # Obter progresso do ClickHouse
    progress_query = """
    SELECT 
        oracle_table,
        ch_table,
        rows_collected,
        total_rows,
        rows_remaining,
        last_id,
        id_column,
        status,
        error
    FROM migration_progress
    FINAL
    WHERE status = 'partial'
    ORDER BY rows_remaining DESC  -- Priorizar tabelas maiores
    """
    
    progress_df = client.query_df(progress_query)
    
    if len(progress_df) == 0:
        print("\n" + "="*60)
        print("✅ MIGRAÇÃO COMPLETA - TODAS AS TABELAS FINALIZADAS!")
        print("="*60)
        break
    
    print(f"[INFO] {len(progress_df)} tabela(s) pendente(s)\n")
    
    batch_inserted = 0
    
    for idx, row in progress_df.iterrows():
        oracle_tbl = row['oracle_table']
        ch_tbl = row['ch_table']
        last_id = row['last_id']
        id_col = row['id_column']
        rows_collected = int(row['rows_collected'])
        rows_remaining = float(row['rows_remaining'])
        
        print(f"\n[{idx + 1}/{len(progress_df)}] {oracle_tbl} -> {ch_tbl}")
        print(f"  Progresso: {rows_collected:,} / {rows_collected + rows_remaining:,.0f} linhas")
        print(f"  Restante: {rows_remaining:,.0f} linhas")
        
        try:
            # Validação pré-inserção
            try:
                df_ch_count = spark.read.format("jdbc") \
                    .option("url", clickhouse_jdbc_url) \
                    .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {ch_tbl}) tmp") \
                    .option("user", clickhouse_user) \
                    .option("password", clickhouse_password) \
                    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
                    .load()
                
                ch_count = int(df_ch_count.collect()[0]['cnt'])
                
                if ch_count != rows_collected:
                    print(f"  [AVISO] Ajustando contador: {ch_count:,} linhas no ClickHouse")
                    rows_collected = ch_count
                    
            except Exception as validate_err:
                print(f"  [AVISO] Validação não disponível: {str(validate_err)[:60]}")
            
            t0 = datetime.now()

            oracle_metadata = get_oracle_column_metadata(oracle_tbl)
            if oracle_metadata:
                print(f"  [INFO] Metadados Oracle carregados: {len(oracle_metadata)} coluna(s)")

            # Construir query Oracle com paginação correta
            if id_col and last_id:
                query = f"""
                (SELECT * FROM (
                    SELECT * FROM {oracle_tbl} 
                    WHERE {id_col} > '{last_id}'
                    ORDER BY {id_col}
                ) WHERE ROWNUM <= {MAX_ROWS}) tmp
                """
            else:
                query = f"""
                (SELECT * FROM 
                    (SELECT a.*, ROWNUM rnum FROM 
                        (SELECT * FROM {oracle_tbl} ORDER BY 1) a
                     WHERE ROWNUM <= {rows_collected + MAX_ROWS})
                 WHERE rnum > {rows_collected}) tmp
                """
            
            # Ler batch do Oracle
            df = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", query) \
                .option("fetchsize", 10000) \
                .load()
            
            # Remover coluna ROWNUM se existir
            if 'rnum' in df.columns:
                df = df.drop('rnum')

            df = apply_oracle_casts(df, oracle_metadata)
            nrows = df.count()
            
            if nrows == 0:
                print(f"  ✅ Migração COMPLETA para esta tabela!")
                stats_complete = pd.DataFrame([{
                    'oracle_table': oracle_tbl,
                    'ch_table': ch_tbl,
                    'rows_collected': rows_collected,
                    'total_rows': float(rows_collected),
                    'rows_remaining': 0.0,
                    'last_id': last_id,
                    'id_column': id_col,
                    'status': 'complete',
                    'error': ''
                }])
                client.insert_df('migration_progress', stats_complete)
                continue
            
            print(f"  Lidas: {nrows:,} linhas")
            
            # [RAW LAYER] Manter dados como estão
            df_clean = df
            
            # Capturar novo último ID
            new_last_id = last_id
            if id_col and id_col in df_clean.columns:
                last_row = df_clean.orderBy(F.col(id_col).desc()).first()
                new_last_id = last_row[id_col] if last_row else last_id
            
            # Escrever no ClickHouse via Spark JDBC
            print(f"  Gravando {nrows:,} linhas no ClickHouse...")
# Antes de gravar, reparticionar para paralelizar a escrita
            num_partitions = 4  # Ajuste conforme necessário

            df_clean.repartition(num_partitions).write \
                .format("jdbc") \
                .option("url", clickhouse_jdbc_url) \
                .option("dbtable", ch_tbl) \
                .option("user", clickhouse_user) \
                .option("password", clickhouse_password) \
                .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
                .option("batchsize", 50000) \
                .option("isolationLevel", "NONE") \
                .option("numPartitions", num_partitions) \
                .mode("append") \
                .save()
            
            # Atualizar progresso
            new_total_collected = rows_collected + nrows
            
            # Contar restante
            if id_col and new_last_id:
                df_remaining = spark.read.format("jdbc") \
                    .options(**jdbc_opts) \
                    .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {oracle_tbl} WHERE {id_col} > '{new_last_id}') tmp") \
                    .load()
                rows_remaining = float(df_remaining.collect()[0]['CNT'])
            else:
                rows_remaining = max(0, rows_remaining - nrows)
            
            dur = (datetime.now() - t0).total_seconds()
            print(f"  ✅ {nrows:,} linhas inseridas em {dur:.2f}s")
            print(f"  Total: {new_total_collected:,} | Restante: {rows_remaining:,.0f}")
            
            # Salvar progresso
            new_status = 'complete' if rows_remaining == 0 else 'partial'
            stats_update = pd.DataFrame([{
                'oracle_table': oracle_tbl,
                'ch_table': ch_tbl,
                'rows_collected': new_total_collected,
                'total_rows': float(new_total_collected + rows_remaining),
                'rows_remaining': rows_remaining,
                'last_id': str(new_last_id),
                'id_column': id_col,
                'status': new_status,
                'error': ''
            }])
            client.insert_df('migration_progress', stats_update)
            
            batch_inserted += nrows
            total_migrated += nrows
            
        except Exception as e:
            msg = str(e)[:300]
            print(f"  ❌ ERRO: {msg}")
            stats_error = pd.DataFrame([{
                'oracle_table': oracle_tbl,
                'ch_table': ch_tbl,
                'rows_collected': rows_collected,
                'total_rows': 0.0,
                'rows_remaining': 0.0,
                'last_id': last_id,
                'id_column': id_col,
                'status': 'error',
                'error': msg[:500]
            }])
            client.insert